# **01a · Download TCGA-COAD Data from GDC (Beginner-Friendly)**
**Goal:** Use the GDC API to create download manifests for TCGA-COAD:
- RNA-seq gene expression counts (HTSeq-Counts)
- Clinical data
- Somatic mutations (MAF)
- DNA methylation (450k)

Then you can use the **gdc-client** to actually download the files into `data_raw/tcga_coad/`.


 **Important notes**
- This notebook only prepares the *lists* and manifests.
- For big downloads, GDC recommends using the official `gdc-client` tool.
- Some data types are controlled-access (need dbGaP + GDC token). We will focus on **open-access** parts.


# **Cell 2 — Settings & imports:**

In [1]:
# === 🔧 SETTINGS ===
PROJECT_ID = "TCGA-COAD"
BASE = "/content/drive/MyDrive/Colorectal_Hippo_Dysbiosis"
RAW_DIR = f"{BASE}/data_raw/tcga_coad"
print("Project:", PROJECT_ID)
print("Save to:", RAW_DIR)

from google.colab import drive
drive.mount('/content/drive')

import os, json, requests
import pandas as pd
from pathlib import Path

Path(RAW_DIR).mkdir(parents=True, exist_ok=True)

GDC_FILES_ENDPOINT = "https://api.gdc.cancer.gov/files"


Project: TCGA-COAD
Save to: /content/drive/MyDrive/Colorectal_Hippo_Dysbiosis/data_raw/tcga_coad
Mounted at /content/drive


# **Cell 3 — Helper to query GDC API and build a manifest**

In [2]:
def gdc_files_query(filters_dict, fields, page_size=200):
    """
    Query GDC files endpoint with JSON filters.

    filters_dict: Python dict describing GDC filters.
    fields: list of field names to request.
    """
    params = {
        "filters": json.dumps({"op": "and", "content": filters_dict}),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": page_size
    }
    r = requests.get(GDC_FILES_ENDPOINT, params=params)
    r.raise_for_status()
    data = r.json()
    hits = data["data"]["hits"]
    print("Found", len(hits), "files")
    return hits

def hits_to_manifest_df(hits):
    """Turn GDC hits into a GDC manifest-style table."""
    rows = []
    for h in hits:
        # fields needed for gdc-client manifest
        rows.append({
            "id": h["file_id"],
            "filename": h["file_name"],
            "md5": h["md5sum"],
            "size": h["file_size"],
            "state": h["state"]
        })
    return pd.DataFrame(rows)

def save_manifest(df, name):
    out = f"{RAW_DIR}/{name}"
    df.to_csv(out, sep="\t", index=False)
    print("Saved manifest ->", out)


In [8]:
# === RNA-seq STAR counts manifest ===
print("Generating RNA-seq manifest using 'STAR - Counts' workflow type.")

# Define filters for 'STAR - Counts' workflow type
filters = [
    {"op": "in", "content": {"field": "cases.project.project_id", "value": [PROJECT_ID]}},
    {"op": "in", "content": {"field": "data_category",           "value": ["Transcriptome Profiling"]}},
    {"op": "in", "content": {"field": "data_type",               "value": ["Gene Expression Quantification"]}},
    {"op": "in", "content": {"field": "analysis.workflow_type",  "value": ["STAR - Counts"]}}
]

# Query GDC for files matching the filters
hits = gdc_files_query(filters, fields=[
    "file_id", "file_name", "md5sum", "file_size", "state"
])

# Convert hits to manifest DataFrame
rna_manifest = hits_to_manifest_df(hits)

display(rna_manifest.head())
save_manifest(rna_manifest, "tcga_coad_star_counts_manifest.tsv")


Generating RNA-seq manifest using 'STAR - Counts' workflow type.
Found 200 files


,id,filename,md5,size,state
0,534054c8-b88f-4b5f-b718-f4ca8484b83c,4d33ae3d-d6e6-47cd-bb5d-52134a7bcf8d.rna_seq.a...,afbb60e489571ba198ef474274c50ace,4222918,released
1,8eaead17-aacf-49b0-9f62-30fb80df9df3,871b21df-9338-419b-a2cc-9cf1552f4da2.rna_seq.a...,6a405361d20c45f13565ef301046c598,4234303,released
2,df5f2430-dabe-4f19-bf81-379b46233ab4,6f043250-acea-4315-8400-a172214c84c7.rna_seq.a...,ca5b179fedd7ef0a61e242877b33c2c0,4211456,released
3,3d1c16a9-2729-4213-9d60-3fc34b4d9911,ef0c5405-76c6-49bd-8463-cdacf603c9cb.rna_seq.a...,230657acb072c49ffdafcde10725f44b,4215495,released
4,c679154e-df11-4b05-9584-76437286d2ee,185b401a-bdc4-46b9-b2a5-7a66938f28f2.rna_seq.a...,a16207bfe3191326996c3f04e51419ff,4213612,released


Saved manifest -> /content/drive/MyDrive/Colorectal_Hippo_Dysbiosis/data_raw/tcga_coad/tcga_coad_star_counts_manifest.tsv


In [4]:
# === Clinical files manifest ===
filters = [
    {"op": "in", "content": {"field": "cases.project.project_id", "value": [PROJECT_ID]}},
    {"op": "in", "content": {"field": "data_category",           "value": ["Clinical"]}}
]

hits = gdc_files_query(filters, fields=[
    "file_id", "file_name", "md5sum", "file_size", "state"
])

clinical_manifest = hits_to_manifest_df(hits)
display(clinical_manifest.head())
save_manifest(clinical_manifest, "tcga_coad_clinical_manifest.tsv")


Found 200 files


,id,filename,md5,size,state
0,7d0a031c-d7fe-4b10-b784-385abb549a62,TCGA-AA-3715.CC2F2E2C-FCC2-4D75-9589-A9EC31FBE...,a674d661a3b32c71df02ca4ebebcecf1,25566,released
1,84d53a9c-0c08-4568-9eab-2d6919511da1,nationwidechildrens.org_omf.TCGA-AA-3531.xml,a1ecd91c780aa32670cd500ca649b871,10728,released
2,f89bffe3-82e6-4753-99ba-b15ffe04ec9e,nationwidechildrens.org_clinical.TCGA-AA-3531.xml,80f2b856eeb1de8d253aef3c5f2ac4c6,29212,released
3,4cddf526-22e1-46aa-9ed1-efd4a49d6cd0,nationwidechildrens.org_omf.TCGA-AA-3860.xml,1a0e998b638dc356bcac97ee065aeb7d,11168,released
4,ce5e7f83-db37-446a-b159-763bc7d7c6e5,TCGA-AA-3814.8521E575-B0FE-42B3-BAAD-107A913A7...,4b59e2212083d14f578887537a798150,9743,released


Saved manifest -> /content/drive/MyDrive/Colorectal_Hippo_Dysbiosis/data_raw/tcga_coad/tcga_coad_clinical_manifest.tsv


In [5]:
# === Somatic mutation (MAF) manifest (may be controlled-access) ===
filters = [
    {"op": "in", "content": {"field": "cases.project.project_id", "value": [PROJECT_ID]}},
    {"op": "in", "content": {"field": "data_category",           "value": ["Simple Nucleotide Variation"]}},
    {"op": "in", "content": {"field": "data_type",               "value": ["Masked Somatic Mutation"]}}
]

hits = gdc_files_query(filters, fields=[
    "file_id", "file_name", "md5sum", "file_size", "state"
])

maf_manifest = hits_to_manifest_df(hits)
display(maf_manifest.head())
save_manifest(maf_manifest, "tcga_coad_maf_manifest.tsv")

print("⚠️ NOTE: These may be controlled-access. You might need a GDC token + dbGaP access to download them.")


Found 200 files


,id,filename,md5,size,state
0,0a7104af-8c37-4492-9e72-f1e3f379e103,2c1ba550-574e-4bc3-a564-87e52ba76961.wxs.aliqu...,90bcd9b86636796cd4ee359b12d33e50,858524,released
1,48a5f19e-ae8b-4055-a5ff-961601aad512,f5ecb5b9-f00d-4d32-a3bd-42f8a168032c.wxs.aliqu...,11c466303107209c41e325cd90ba5de6,22783,released
2,976e5a9e-dfe4-42e0-95fe-d7a33c8eecbc,d66378d1-4ec6-468f-8127-094989b2be3d.wxs.aliqu...,97124a969100de1709514298b965319b,42694,released
3,9e9b5bb4-ee5a-48e4-8765-e1fc322c8dc3,530fb799-3e21-4dfe-9b8b-2b39bd6b5d8b.wxs.aliqu...,3cad22ad884c895a61cdcd6fa2c49d9a,37525,released
4,ff94da29-7fef-45ad-b73c-6c6f8acd8bfa,35b1f6da-e7d4-4eb0-8fed-83ce63958181.wxs.aliqu...,628dac3fbd8d78720705d9f944f260fe,57088,released


Saved manifest -> /content/drive/MyDrive/Colorectal_Hippo_Dysbiosis/data_raw/tcga_coad/tcga_coad_maf_manifest.tsv
⚠️ NOTE: These may be controlled-access. You might need a GDC token + dbGaP access to download them.


In [6]:
# === DNA methylation (450k) manifest ===
filters = [
    {"op": "in", "content": {"field": "cases.project.project_id", "value": [PROJECT_ID]}},
    {"op": "in", "content": {"field": "data_category",           "value": ["DNA Methylation"]}},
    {"op": "in", "content": {"field": "platform",                "value": ["Illumina Human Methylation 450"]}}
]

hits = gdc_files_query(filters, fields=[
    "file_id", "file_name", "md5sum", "file_size", "state"
])

meth_manifest = hits_to_manifest_df(hits)
display(meth_manifest.head())
save_manifest(meth_manifest, "tcga_coad_methylation_450k_manifest.tsv")


Found 200 files


,id,filename,md5,size,state
0,e917c10c-89be-4ca4-90ee-d0bcf301167d,ea34e993-45b5-4830-88b4-dec2051b6daa_noid_Grn....,7eb25d7e7a0ddd08a13855b588475f7b,8095206,released
1,c294c7de-8600-4c1d-a08c-b4c6be7622b6,9f1bc9a7-9808-4c3b-911b-7313f160b9df_noid_Red....,41b03649ad960e18b06b88e7e38dad90,8095236,released
2,663715ee-7027-492a-84be-59ef63d4ca9b,16dc9702-9a7d-4d9c-8c04-fd2110e00b15.methylati...,38230d4dbca251aa964252e9ed6440cc,13147860,released
3,03df8196-a3e2-4eff-80ca-7dc897e8394b,c6cfad8c-85f2-41b0-b997-155e8b696bb8.methylati...,bd7aa3013d381f4471c57dcc92ccd681,13167426,released
4,a059164d-4a9f-4264-89f7-be83dc22fc0f,16dc9702-9a7d-4d9c-8c04-fd2110e00b15_noid_Grn....,e19a422fbc63feb3cdb1c1c64196bf28,8095212,released


Saved manifest -> /content/drive/MyDrive/Colorectal_Hippo_Dysbiosis/data_raw/tcga_coad/tcga_coad_methylation_450k_manifest.tsv
